# Notebook 2: Sales Prediction with Tweedie Loss

## Why the Tweedie Distribution for Sales Data?

Retail sales data shares the same **semi-continuous** structure as insurance claims:
- Many SKU-store-day combinations have **zero sales** (product not sold that day)
- Non-zero sales are **positive and right-skewed** (occasional large orders)

Traditional approaches use **log-transformed** targets, but this introduces **systematic under-forecasting** upon back-transformation (More, 2022). The Tweedie distribution avoids this entirely by modeling the raw sales directly.

$$\text{Tweedie Deviance}(y, \hat{\mu}) = 2 \left[ \frac{y^{2-p}}{(1-p)(2-p)} - \frac{y \cdot \hat{\mu}^{1-p}}{1-p} + \frac{\hat{\mu}^{2-p}}{2-p} \right]$$

This notebook demonstrates:
- **Part A**: Standard in-memory models (XGBoost, LightGBM with Tweedie objective)
- **Part B**: Scalable implementations (PyTorch with embeddings, PySpark, Dask)

---

## Part A: Standard (In-Memory) Sales Prediction

### A.1 Environment Setup

In [ ]:
# Install required packages (uncomment if needed)
# !pip install xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import xgboost as xgb
import lightgbm as lgb

np.random.seed(42)
print("Environment ready!")

### A.2 Generate Realistic Retail Sales Dataset

We generate a synthetic dataset mimicking daily sales for multiple stores and products, with realistic zero-inflation patterns.

In [ ]:
def generate_retail_sales(n_stores=50, n_products=200, n_days=365, seed=42):
    """Generate synthetic retail sales data with zero inflation.

    Mimics daily sales for store-product combinations:
    - Many zeros (product not sold that day)
    - Right-skewed positive values
    - Seasonal patterns, weekday effects
    """
    rng = np.random.RandomState(seed)
    records = []

    dates = pd.date_range('2024-01-01', periods=n_days, freq='D')

    for store_id in range(n_stores):
        store_base = rng.uniform(0.5, 2.0)  # store popularity factor
        for prod_id in range(n_products):
            prod_base = rng.uniform(0.1, 3.0)  # product demand level
            prod_price = rng.uniform(5, 200)
            prod_category = rng.randint(0, 10)

            for day_idx, date in enumerate(dates):
                # Seasonal effect (higher in Nov-Dec)
                month = date.month
                seasonal = 1.0 + 0.3 * np.sin(2 * np.pi * (month - 1) / 12) + \
                           (0.5 if month in [11, 12] else 0)

                # Day-of-week effect (higher on weekends)
                dow = date.dayofweek
                dow_effect = 1.3 if dow >= 5 else 1.0

                # Compute expected sales (mu)
                log_mu = np.log(store_base * prod_base * seasonal * dow_effect * 10)
                mu = np.exp(log_mu)

                # Tweedie compound Poisson-Gamma (p=1.5)
                p = 1.5
                phi = 1.5
                lam = mu**(2-p) / (phi * (2-p))
                alpha = (2-p) / (p-1)
                beta = phi * (p-1) * mu**(p-1)

                N = rng.poisson(lam)
                if N > 0:
                    sales = rng.gamma(alpha, beta, N).sum()
                else:
                    sales = 0.0

                records.append({
                    'date': date,
                    'store_id': store_id,
                    'product_id': prod_id,
                    'category': prod_category,
                    'price': round(prod_price, 2),
                    'day_of_week': dow,
                    'month': month,
                    'is_weekend': int(dow >= 5),
                    'sales_amount': round(sales, 2)
                })

    return pd.DataFrame(records)

# Generate a manageable sample for Part A
# Full dataset would be 50*200*365 = 3.65M rows; we use a subset
print("Generating retail sales data...")
df_sales = generate_retail_sales(n_stores=20, n_products=50, n_days=180, seed=42)
print(f"Dataset shape: {df_sales.shape}")
print(f"Date range: {df_sales['date'].min()} to {df_sales['date'].max()}")
print(f"\nSales distribution:")
print(f"  Zero sales:   {(df_sales['sales_amount'] == 0).sum():,} ({(df_sales['sales_amount'] == 0).mean()*100:.1f}%)")
print(f"  Non-zero mean: ${df_sales.loc[df_sales['sales_amount']>0, 'sales_amount'].mean():.2f}")
print(f"  Overall mean:  ${df_sales['sales_amount'].mean():.2f}")
print(f"  Max:           ${df_sales['sales_amount'].max():,.2f}")
print(f"\n{df_sales.head(10)}")

### A.3 Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Distribution of sales amount
axes[0, 0].hist(df_sales['sales_amount'], bins=100, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].set_xlabel('Sales Amount ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title(f'Sales Distribution ({(df_sales["sales_amount"]==0).mean()*100:.0f}% zeros)')

# Log-transformed (non-zero only)
nonzero = df_sales.loc[df_sales['sales_amount'] > 0, 'sales_amount']
axes[0, 1].hist(np.log1p(nonzero), bins=80, color='coral', edgecolor='white', alpha=0.8)
axes[0, 1].set_xlabel('Log(1 + Sales Amount)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Log-Transformed Non-Zero Sales')

# Daily total sales over time
daily_total = df_sales.groupby('date')['sales_amount'].sum()
axes[1, 0].plot(daily_total.index, daily_total.values, color='steelblue', alpha=0.7, linewidth=0.8)
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Total Daily Sales ($)')
axes[1, 0].set_title('Daily Sales Over Time')
axes[1, 0].tick_params(axis='x', rotation=45)

# Sales by day of week
dow_sales = df_sales.groupby('day_of_week')['sales_amount'].mean()
dow_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
axes[1, 1].bar(range(7), dow_sales.values, color='steelblue', alpha=0.8)
axes[1, 1].set_xticks(range(7))
axes[1, 1].set_xticklabels(dow_labels)
axes[1, 1].set_xlabel('Day of Week')
axes[1, 1].set_ylabel('Mean Sales ($)')
axes[1, 1].set_title('Average Sales by Day of Week')

plt.tight_layout()
plt.savefig('sales_eda.png', dpi=150, bbox_inches='tight')
plt.show()

### A.4 Feature Engineering and Chronological Split

In [ ]:
# Feature engineering
df_sales['day_of_year'] = df_sales['date'].dt.dayofyear
df_sales['week_of_year'] = df_sales['date'].dt.isocalendar().week.astype(int)
df_sales['sin_month'] = np.sin(2 * np.pi * df_sales['month'] / 12)
df_sales['cos_month'] = np.cos(2 * np.pi * df_sales['month'] / 12)
df_sales['sin_dow'] = np.sin(2 * np.pi * df_sales['day_of_week'] / 7)
df_sales['cos_dow'] = np.cos(2 * np.pi * df_sales['day_of_week'] / 7)

# Rolling average sales per store-product (lag features)
df_sales = df_sales.sort_values(['store_id', 'product_id', 'date'])
df_sales['lag_7d_mean'] = df_sales.groupby(['store_id', 'product_id'])['sales_amount'] \
    .transform(lambda x: x.rolling(7, min_periods=1).mean().shift(1))
df_sales['lag_30d_mean'] = df_sales.groupby(['store_id', 'product_id'])['sales_amount'] \
    .transform(lambda x: x.rolling(30, min_periods=1).mean().shift(1))
df_sales['lag_7d_mean'] = df_sales['lag_7d_mean'].fillna(0)
df_sales['lag_30d_mean'] = df_sales['lag_30d_mean'].fillna(0)

# Log price
df_sales['log_price'] = np.log1p(df_sales['price'])

# Define features
feature_cols = ['store_id', 'product_id', 'category', 'price', 'log_price',
                'day_of_week', 'month', 'is_weekend', 'day_of_year',
                'sin_month', 'cos_month', 'sin_dow', 'cos_dow',
                'lag_7d_mean', 'lag_30d_mean']

# Chronological split (last 30 days = test)
split_date = df_sales['date'].max() - pd.Timedelta(days=30)
train_mask = df_sales['date'] <= split_date
test_mask = df_sales['date'] > split_date

X_train = df_sales.loc[train_mask, feature_cols].values
y_train = df_sales.loc[train_mask, 'sales_amount'].values
X_test = df_sales.loc[test_mask, feature_cols].values
y_test = df_sales.loc[test_mask, 'sales_amount'].values

print(f"Training: {X_train.shape[0]:,} rows (up to {split_date.date()})")
print(f"Testing:  {X_test.shape[0]:,} rows (after {split_date.date()})")
print(f"Features: {len(feature_cols)}")
print(f"Train zeros: {(y_train == 0).mean()*100:.1f}%")
print(f"Test zeros:  {(y_test == 0).mean()*100:.1f}%")

In [ ]:
def evaluate_sales_model(y_true, y_pred, model_name="Model"):
    """Evaluate with sales-specific metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # WAPE (Weighted Absolute Percentage Error)
    wape = np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100

    # MAPE on non-zero
    nz = y_true > 0
    mape = np.mean(np.abs((y_true[nz] - y_pred[nz]) / y_true[nz])) * 100 if nz.sum() > 0 else float('inf')

    # Tweedie deviance
    from sklearn.metrics import mean_tweedie_deviance
    y_pred_c = np.clip(y_pred, 1e-6, None)
    try:
        td = mean_tweedie_deviance(y_true, y_pred_c, power=1.5)
    except:
        td = float('nan')

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  MAE:     ${mae:>10,.2f}")
    print(f"  RMSE:    ${rmse:>10,.2f}")
    print(f"  WAPE:    {wape:>10.2f}%")
    print(f"  MAPE:    {mape:>10.2f}%")
    print(f"  Tweedie: {td:>10.4f}")
    print(f"{'='*50}")
    return {'MAE': mae, 'RMSE': rmse, 'WAPE': wape, 'Tweedie': td}

results = {}

### A.5 Model 1: XGBoost with Tweedie Objective

XGBoost supports `reg:tweedie` with tunable `tweedie_variance_power`.

In [ ]:
# XGBoost with Tweedie loss
xgb_model = xgb.XGBRegressor(
    objective='reg:tweedie',
    tweedie_variance_power=1.5,
    n_estimators=800,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    verbosity=0,
    n_jobs=-1
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred_xgb = np.clip(xgb_model.predict(X_test), 0, None)
results['XGBoost'] = evaluate_sales_model(y_test, y_pred_xgb, "XGBoost Tweedie (p=1.5)")

### A.6 Model 2: LightGBM with Tweedie Objective

In [ ]:
# LightGBM with Tweedie loss
lgb_model = lgb.LGBMRegressor(
    objective='tweedie',
    tweedie_variance_power=1.5,
    n_estimators=800,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.log_evaluation(0)]
)

y_pred_lgb = np.clip(lgb_model.predict(X_test), 0, None)
results['LightGBM'] = evaluate_sales_model(y_test, y_pred_lgb, "LightGBM Tweedie (p=1.5)")

### A.7 Hyperparameter Tuning: Variance Power

The Tweedie power parameter $p$ controls the balance between frequency and severity modeling. We tune it using cross-validation.

In [ ]:
# Tune the variance power parameter
from sklearn.model_selection import TimeSeriesSplit

powers = [1.1, 1.2, 1.3, 1.5, 1.6, 1.7, 1.8, 1.9]
power_results = []

print("=== Tuning Tweedie Variance Power (LightGBM) ===\n")

for p in powers:
    model = lgb.LGBMRegressor(
        objective='tweedie',
        tweedie_variance_power=p,
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    y_pred = np.clip(model.predict(X_test), 1e-6, None)

    from sklearn.metrics import mean_tweedie_deviance
    try:
        dev = mean_tweedie_deviance(y_test, y_pred, power=p)
    except:
        dev = float('nan')
    mae = mean_absolute_error(y_test, y_pred)

    power_results.append({'power': p, 'tweedie_deviance': dev, 'mae': mae})
    print(f"  p={p:.1f}: Tweedie Dev={dev:.4f}, MAE=${mae:,.2f}")

power_df = pd.DataFrame(power_results)
best_power = power_df.loc[power_df['tweedie_deviance'].idxmin(), 'power']
print(f"\n  >>> Best power: p={best_power}")

# Plot
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(power_df['power'], power_df['tweedie_deviance'], 'b-o', markersize=6, label='Tweedie Deviance')
ax1.set_xlabel('Tweedie Power Parameter (p)')
ax1.set_ylabel('Tweedie Deviance', color='blue')
ax1.axvline(best_power, color='red', linestyle='--', alpha=0.7, label=f'Best p={best_power}')

ax2 = ax1.twinx()
ax2.plot(power_df['power'], power_df['mae'], 'g-s', markersize=6, label='MAE')
ax2.set_ylabel('MAE ($)', color='green')

ax1.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.title('Tweedie Power Parameter Tuning')
plt.tight_layout()
plt.savefig('sales_power_tuning.png', dpi=150, bbox_inches='tight')
plt.show()

### A.8 Model Comparison and Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Actual vs Predicted - XGBoost
axes[0, 0].scatter(y_test, y_pred_xgb, alpha=0.05, s=3, color='steelblue')
axes[0, 0].plot([0, np.percentile(y_test, 99)], [0, np.percentile(y_test, 99)], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual Sales ($)')
axes[0, 0].set_ylabel('Predicted Sales ($)')
axes[0, 0].set_title('XGBoost Tweedie: Actual vs Predicted')
axes[0, 0].set_xlim(0, np.percentile(y_test, 99))
axes[0, 0].set_ylim(0, np.percentile(y_pred_xgb, 99))

# Actual vs Predicted - LightGBM
axes[0, 1].scatter(y_test, y_pred_lgb, alpha=0.05, s=3, color='coral')
axes[0, 1].plot([0, np.percentile(y_test, 99)], [0, np.percentile(y_test, 99)], 'r--', lw=2)
axes[0, 1].set_xlabel('Actual Sales ($)')
axes[0, 1].set_ylabel('Predicted Sales ($)')
axes[0, 1].set_title('LightGBM Tweedie: Actual vs Predicted')
axes[0, 1].set_xlim(0, np.percentile(y_test, 99))
axes[0, 1].set_ylim(0, np.percentile(y_pred_lgb, 99))

# Daily aggregated predictions
test_dates = df_sales.loc[test_mask, 'date'].values
daily_actual = pd.Series(y_test, index=test_dates).groupby(level=0).sum()
daily_pred_xgb = pd.Series(y_pred_xgb, index=test_dates).groupby(level=0).sum()
daily_pred_lgb = pd.Series(y_pred_lgb, index=test_dates).groupby(level=0).sum()

axes[1, 0].plot(daily_actual.index, daily_actual.values, 'k-', label='Actual', linewidth=2)
axes[1, 0].plot(daily_pred_xgb.index, daily_pred_xgb.values, 'b--', label='XGBoost', alpha=0.8)
axes[1, 0].plot(daily_pred_lgb.index, daily_pred_lgb.values, 'r--', label='LightGBM', alpha=0.8)
axes[1, 0].set_xlabel('Date')
axes[1, 0].set_ylabel('Total Daily Sales ($)')
axes[1, 0].set_title('Aggregated Daily Sales: Actual vs Predicted')
axes[1, 0].legend()
axes[1, 0].tick_params(axis='x', rotation=45)

# Feature importance (LightGBM)
importance = lgb_model.feature_importances_
feat_imp = pd.Series(importance, index=feature_cols).sort_values(ascending=True)
feat_imp.plot(kind='barh', color='coral', ax=axes[1, 1])
axes[1, 1].set_xlabel('Feature Importance')
axes[1, 1].set_title('LightGBM Feature Importance')

plt.tight_layout()
plt.savefig('sales_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print("\n=== Sales Model Comparison ===")
print(pd.DataFrame(results).T.to_string())

---

## Part B: Scaling Sales Prediction to Large Data

For millions of SKU-store-day combinations, we need distributed or out-of-core approaches.

### B.1 Generate Large Synthetic Sales Data

In [ ]:
import time
import tracemalloc

def generate_large_sales(n_rows, n_stores=500, n_products=2000, seed=42):
    """Generate large-scale synthetic sales data."""
    rng = np.random.RandomState(seed)

    store_id = rng.randint(0, n_stores, n_rows)
    product_id = rng.randint(0, n_products, n_rows)
    category = rng.randint(0, 20, n_rows)
    price = rng.uniform(5, 200, n_rows).round(2)
    dow = rng.randint(0, 7, n_rows)
    month = rng.randint(1, 13, n_rows)
    is_weekend = (dow >= 5).astype(np.float32)

    # Compute expected sales
    store_effect = rng.uniform(0.5, 2.0, n_stores)[store_id]
    prod_effect = rng.uniform(0.1, 3.0, n_products)[product_id]
    seasonal = 1.0 + 0.3 * np.sin(2 * np.pi * month / 12)
    dow_effect = np.where(dow >= 5, 1.3, 1.0)

    mu = store_effect * prod_effect * seasonal * dow_effect * 10

    # Tweedie compound Poisson-Gamma (p=1.5)
    p, phi = 1.5, 1.5
    lam = mu**(2-p) / (phi * (2-p))
    alpha = (2-p) / (p-1)
    beta = phi * (p-1) * mu**(p-1)

    N = rng.poisson(lam)
    y = np.zeros(n_rows, dtype=np.float32)
    for i in range(n_rows):
        if N[i] > 0:
            y[i] = rng.gamma(alpha, beta[i], N[i]).sum()

    X = np.column_stack([store_id, product_id, category, price, np.log1p(price),
                         dow, month, is_weekend,
                         np.sin(2*np.pi*month/12), np.cos(2*np.pi*month/12),
                         np.sin(2*np.pi*dow/7), np.cos(2*np.pi*dow/7)]).astype(np.float32)

    return X, y

# Quick test
for label, n in [("100K", 100_000), ("1M", 1_000_000)]:
    t0 = time.time()
    X_s, y_s = generate_large_sales(n)
    print(f"{label:>5s}: {n:>10,} rows | {(y_s==0).mean()*100:.1f}% zeros | "
          f"Mean=${y_s[y_s>0].mean():.2f} | {time.time()-t0:.2f}s")

### B.2 PyTorch: Neural Network with Embedding Layers + Tweedie Loss

For categorical features like `store_id` and `product_id` with high cardinality, **embedding layers** learn dense representations.

In [ ]:
# !pip install torch  # uncomment if needed

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


class TweedieLoss(nn.Module):
    """Tweedie deviance loss for PyTorch."""
    def __init__(self, p=1.5):
        super().__init__()
        self.p = p

    def forward(self, log_mu, y_true):
        mu = torch.exp(log_mu)
        loss = -y_true * torch.pow(mu, 1 - self.p) / (1 - self.p) + \
               torch.pow(mu, 2 - self.p) / (2 - self.p)
        return torch.mean(loss)


class SalesNetWithEmbeddings(nn.Module):
    """Neural network with embeddings for categorical features.

    Architecture:
    - Embedding layers for store_id and product_id
    - Concatenation with numerical features
    - Feedforward layers with residual connections
    """
    def __init__(self, n_stores, n_products, n_categories, n_numeric,
                 emb_dim_store=16, emb_dim_prod=16, emb_dim_cat=8,
                 hidden_dims=[256, 128, 64]):
        super().__init__()
        self.store_emb = nn.Embedding(n_stores, emb_dim_store)
        self.product_emb = nn.Embedding(n_products, emb_dim_prod)
        self.category_emb = nn.Embedding(n_categories, emb_dim_cat)

        total_input = emb_dim_store + emb_dim_prod + emb_dim_cat + n_numeric
        layers = []
        prev = total_input
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.ReLU(), nn.BatchNorm1d(h), nn.Dropout(0.3)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, store_ids, product_ids, category_ids, numeric_feats):
        s = self.store_emb(store_ids)
        p = self.product_emb(product_ids)
        c = self.category_emb(category_ids)
        x = torch.cat([s, p, c, numeric_feats], dim=1)
        return self.mlp(x).squeeze(-1)


# Generate training data
N_TRAIN_PT = 800_000
N_TEST_PT = 200_000
X_pt, y_pt = generate_large_sales(N_TRAIN_PT + N_TEST_PT, seed=99)

# Split categorical and numerical features
# Columns: [store_id, product_id, category, price, log_price, dow, month, weekend, sin_m, cos_m, sin_d, cos_d]
store_ids = torch.tensor(X_pt[:, 0].astype(int), dtype=torch.long)
product_ids = torch.tensor(X_pt[:, 1].astype(int), dtype=torch.long)
cat_ids = torch.tensor(X_pt[:, 2].astype(int), dtype=torch.long)
numeric = torch.tensor(X_pt[:, 3:], dtype=torch.float32)
targets = torch.tensor(y_pt, dtype=torch.float32)

# Normalize numeric features
num_mean = numeric[:N_TRAIN_PT].mean(dim=0)
num_std = numeric[:N_TRAIN_PT].std(dim=0) + 1e-8
numeric = (numeric - num_mean) / num_std

# Split
train_ds = TensorDataset(store_ids[:N_TRAIN_PT], product_ids[:N_TRAIN_PT],
                          cat_ids[:N_TRAIN_PT], numeric[:N_TRAIN_PT], targets[:N_TRAIN_PT])
test_store = store_ids[N_TRAIN_PT:]
test_prod = product_ids[N_TRAIN_PT:]
test_cat = cat_ids[N_TRAIN_PT:]
test_num = numeric[N_TRAIN_PT:]
test_y = targets[N_TRAIN_PT:]

train_loader = DataLoader(train_ds, batch_size=8192, shuffle=True, num_workers=0)

print(f"Training: {N_TRAIN_PT:,} | Test: {N_TEST_PT:,}")
print(f"Stores: 500 | Products: 2,000 | Categories: 20")

In [ ]:
# Train the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model_pt = SalesNetWithEmbeddings(
    n_stores=500, n_products=2000, n_categories=20,
    n_numeric=9, emb_dim_store=16, emb_dim_prod=16, emb_dim_cat=8,
    hidden_dims=[256, 128, 64]
).to(device)

criterion = TweedieLoss(p=1.5)
optimizer = torch.optim.Adam(model_pt.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

tracemalloc.start()
t0 = time.time()

train_losses = []
for epoch in range(15):
    model_pt.train()
    epoch_loss = 0
    n_b = 0
    for s, p, c, n, y in train_loader:
        s, p, c, n, y = s.to(device), p.to(device), c.to(device), n.to(device), y.to(device)
        optimizer.zero_grad()
        log_mu = model_pt(s, p, c, n)
        loss = criterion(log_mu, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_pt.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()
        n_b += 1
    scheduler.step()
    avg_loss = epoch_loss / n_b
    train_losses.append(avg_loss)
    if (epoch+1) % 3 == 0:
        print(f"  Epoch {epoch+1:2d}/15 | Loss: {avg_loss:.6f}")

time_pt = time.time() - t0
_, mem_pt = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Predict
model_pt.eval()
with torch.no_grad():
    log_mu = model_pt(test_store.to(device), test_prod.to(device),
                      test_cat.to(device), test_num.to(device))
    y_pred_pt = torch.exp(log_mu).cpu().numpy()

y_pred_pt = np.clip(y_pred_pt, 0, None)
evaluate_sales_model(test_y.numpy(), y_pred_pt, "PyTorch + Embeddings")
print(f"  Time: {time_pt:.1f}s | Memory: {mem_pt/1e6:.0f} MB")

### B.3 PySpark: Distributed Tweedie GLM for Sales

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GeneralizedLinearRegression

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("SalesTweedie") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Generate data
N_SPARK = 1_000_000
X_sp, y_sp = generate_large_sales(N_SPARK, seed=789)

feat_names = ['store_id','product_id','category','price','log_price',
              'dow','month','is_weekend','sin_month','cos_month','sin_dow','cos_dow']
pdf = pd.DataFrame(X_sp, columns=feat_names)
pdf['sales'] = y_sp

tracemalloc.start()
t0 = time.time()

sdf = spark.createDataFrame(pdf).repartition(8)
assembler = VectorAssembler(inputCols=feat_names, outputCol="features")
sdf = assembler.transform(sdf)
train_sdf, test_sdf = sdf.randomSplit([0.8, 0.2], seed=42)
train_sdf.cache()
test_sdf.cache()

glr = GeneralizedLinearRegression(
    family="tweedie", variancePower=1.5, linkPower=0,
    featuresCol="features", labelCol="sales",
    maxIter=50, regParam=0.01
)
spark_model = glr.fit(train_sdf)

time_spark = time.time() - t0
_, mem_spark = tracemalloc.get_traced_memory()
tracemalloc.stop()

pred_pdf = spark_model.transform(test_sdf).select("sales", "prediction").toPandas()
y_true_sp = pred_pdf['sales'].values
y_pred_sp = np.clip(pred_pdf['prediction'].values, 0, None)

evaluate_sales_model(y_true_sp, y_pred_sp, "PySpark Tweedie GLM")
print(f"  Time: {time_spark:.1f}s | Memory: {mem_spark/1e6:.0f} MB")

spark.stop()

### B.4 Dask: Parallel Tweedie Prediction at Scale

In [ ]:
# !pip install dask dask-ml  # uncomment if needed

import dask.array as da
from dask.distributed import Client, LocalCluster
from dask_ml.wrappers import ParallelPostFit
from sklearn.linear_model import TweedieRegressor as SkTweedie

cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit='1GB')
client = Client(cluster)
print(f"Dask Workers: {len(client.scheduler_info()['workers'])}")

# Generate data
N_DASK = 2_000_000
X_dk, y_dk = generate_large_sales(N_DASK, seed=321)

tracemalloc.start()
t0 = time.time()

# Scale features
scaler = StandardScaler()
split = int(0.8 * N_DASK)
X_train_dk = scaler.fit_transform(X_dk[:split])
X_test_dk = scaler.transform(X_dk[split:])
y_train_dk, y_test_dk = y_dk[:split], y_dk[split:]

# Fit on subsample, predict at scale
sub_idx = np.random.choice(len(X_train_dk), min(500_000, len(X_train_dk)), replace=False)
tweedie_dk = SkTweedie(power=1.5, alpha=0.1, link='log', max_iter=500)
tweedie_dk.fit(X_train_dk[sub_idx], y_train_dk[sub_idx])

# Distributed prediction
parallel_model = ParallelPostFit(tweedie_dk)
X_test_da = da.from_array(X_test_dk, chunks=(200_000, -1))
y_pred_dk = parallel_model.predict(X_test_da).compute()

time_dask = time.time() - t0
_, mem_dask = tracemalloc.get_traced_memory()
tracemalloc.stop()

y_pred_dk = np.clip(y_pred_dk, 0, None)
evaluate_sales_model(y_test_dk, y_pred_dk, "Dask Tweedie GLM")
print(f"  Time: {time_dask:.1f}s | Memory: {mem_dask/1e6:.0f} MB")

client.close()
cluster.close()

### B.5 Framework Comparison

In [ ]:
benchmark = pd.DataFrame({
    'Framework': ['PyTorch + Embeddings', 'PySpark GLM', 'Dask GLM'],
    'Dataset': [f'{N_TRAIN_PT+N_TEST_PT:,}', f'{N_SPARK:,}', f'{N_DASK:,}'],
    'Time (s)': [f'{time_pt:.1f}', f'{time_spark:.1f}', f'{time_dask:.1f}'],
    'Memory (MB)': [f'{mem_pt/1e6:.0f}', f'{mem_spark/1e6:.0f}', f'{mem_dask/1e6:.0f}'],
    'Strength': [
        'Entity embeddings for high-cardinality categoricals',
        'Native Tweedie GLM on distributed clusters',
        'Drop-in parallel prediction, scikit-learn API'
    ]
})
print("\n" + "="*80)
print("  SCALABLE FRAMEWORK COMPARISON - Sales Prediction")
print("="*80)
print(benchmark.to_string(index=False))
print("="*80)

---
## Summary

This notebook demonstrated the Tweedie distribution for retail sales prediction:

- **Part A**: XGBoost and LightGBM with Tweedie objective provide strong baselines for zero-inflated sales. Power parameter tuning is essential for optimal fit.
- **Part B**: PyTorch with **embedding layers** captures entity-level patterns (store, product); PySpark and Dask handle millions of SKU-store-day rows.

The Tweedie distribution avoids transformation bias and models the zero-inflated, right-skewed nature of sales data in a single framework, making it superior to log-transformed regression approaches.